### Color Extraction from Image

This section will allow you to upload an image and extract its dominant colors. We will use `Pillow` for image processing and `scikit-learn`'s `KMeans` algorithm for color clustering.

In [ ]:
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans
from IPython.display import display, HTML
from google.colab import files
import io
import ipywidgets as widgets
from ipywidgets import interact, IntSlider


uploaded_pil_image = None
uploaded_filename = None

def get_dominant_colors(image_source, num_colors=5):

    if isinstance(image_source, Image.Image):
        img = image_source.copy()
    else:
        img = Image.open(image_source)

    img.thumbnail((200, 200))

    # Convert image to RGB (discard alpha channel if present)
    img = img.convert('RGB')

    # Convert image data to a numpy array
    img_array = np.array(img)


    pixels = img_array.reshape(-1, 3)


    kmeans = KMeans(n_clusters=num_colors, random_state=42, n_init=10)
    kmeans.fit(pixels)


    dominant_colors_rgb = kmeans.cluster_centers_.astype(int)

    return img, dominant_colors_rgb

def display_colors(image, colors):
    print("Original Image:")
    display(image)

    print(f"\nDominant Colors ({len(colors)}):")
    color_swatches_html = ""
    for rgb in colors:
        hex_color = '#%02x%02x%02x' % tuple(rgb)
        rgb_str = f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"
        color_swatches_html += f"<div style='display:inline-block; width:100px; height:30px; background-color:{hex_color}; border:1px solid #ccc; margin:5px; text-align:center; line-height:30px; color:black;'>{hex_color}<br>{rgb_str}</div>"
    display(HTML(color_swatches_html))

def interactive_color_extractor(num_colors=5):
    global uploaded_pil_image

    if uploaded_pil_image is None:
        print("Please upload an image first by running the cell.")
        return

    print(f"Extracting {num_colors} dominant colors...")
    try:

        original_image_for_display, dominant_colors = get_dominant_colors(uploaded_pil_image, num_colors)
        display_colors(original_image_for_display, dominant_colors)
    except Exception as e:
        print(f"Error processing image with {num_colors} colors: {e}")
        print("Please ensure the uploaded file was a valid image.")

# --- File Upload ---
print("Please upload an image file (e.g., JPG, PNG):")
uploaded_files = files.upload()

if uploaded_files:
    for filename, content in uploaded_files.items():
        print(f"Uploaded file: {filename}")
        uploaded_filename = filename
        # Store the PIL Image object globally for repeated use by the slider
        uploaded_pil_image = Image.open(io.BytesIO(content))
        print("Image uploaded successfully! Use the slider below to adjust the number of dominant colors.")
        break # Assuming only one file is uploaded

    if uploaded_pil_image:
        print("\nAdjust the slider to change the number of dominant colors:")

        num_colors_slider = IntSlider(min=1, max=20, step=1, value=5, description='Num Colors:')

        interact(interactive_color_extractor, num_colors=num_colors_slider)
else:
    print("No file uploaded.")

Please upload an image file (e.g., JPG, PNG):


Saving 38f3a469ae3736c89d9f2fdff69fdc23.jpg to 38f3a469ae3736c89d9f2fdff69fdc23 (1).jpg
Uploaded file: 38f3a469ae3736c89d9f2fdff69fdc23 (1).jpg
Image uploaded successfully! Use the slider below to adjust the number of dominant colors.

Adjust the slider to change the number of dominant colors:


interactive(children=(IntSlider(value=5, description='Num Colors:', max=20, min=1), Output()), _dom_classes=('…